# Meausure energy-energy correlators with Omnifold measurement

This notebook implements a measurement of the two-point energy correlator observable (EEC) using the ATLAS Omnifold Z+jets measurement. The observable is first examined using the "pseudo-measurement" and statistical agreement with the target truth pseudodata distribution is checked. Then the results of the measurement on data are presented.

Note given that re-calculating the energy products and angular coordinates of track pairs is relatively inexpensive once parallelized, this notebook does not pre-compute these results and write them to disk. Instead the histogramming is the compute expensive step. For better performance, both steps are parallelized using dask delayed. See `eec_utils.py` for details.

This notebook takes approximately XXX minutes to run on a CPU node on the Perlmutter supercomputer.

## Preliminaries: load all data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import pickle
import numpy as np
import awkward as ak
import uproot
import pandas as pd
import dask.distributed as distributed

import eec_utils as eec
import visualize as vis

In [3]:
# ROOT file paths
staging_dir = "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data"
tree_name = "OmniTree"

madgraph_path = f"{staging_dir}/madgraph_events.root"
pseudodata_path = f"{staging_dir}/truth_pseudodata_events.root"
hv_path = f"{staging_dir}/hv_events.root"
sherpa_paths = [
    f"{staging_dir}/sherpa/sherpa_events_part1.root",
    f"{staging_dir}/sherpa/sherpa_events_part2.root",
]

In [4]:
# Load event weights for pseudodata measurement
pseudodata_weights = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/pseudodata_weights.h5"
)
pseudodata_hv_weights = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/pseudodata_hv_weights.h5"
)
truth_pseudodata_weights = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/truth_pseudodata_weights.h5"
)

# Load event weights for data measurement
data_weights = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/data_weights.h5"
)
data_hv_weights = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/data_hv_weights.h5"
)
madgraph_weights = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/madgraph_weights.h5"
)
sherpa_weights1 = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/sherpa/sherpa_weights_part1.h5"
)
sherpa_weights2 = pd.read_hdf(
    "/pscratch/sd/k/kgreif/zjets_plot_staging/final_public_data/sherpa/sherpa_weights_part2.h5"
)

## Measurement with Pseudodata

In [5]:
# Create dictionary containined all of the weight vectors needed for the pseudodata measurement
nominal_weights = madgraph_weights["weights_nominal"].to_numpy()
n_init_weights = [
    name for name in pseudodata_weights.keys() if "weights_ensemble_" in name
]
dbootstrap_weights = [
    name for name in pseudodata_weights.keys() if "weights_bootstrap_data_" in name
]
mcbootstrap_train_weights = [
    name for name in pseudodata_weights.keys() if "weights_bootstrap_mc_" in name
]
pd_weights = {
    "prior": madgraph_weights[
        "weights_nominal"
    ],  # Also include weights for plotting the "prior" used in the measurement
    "nominal": nominal_weights,
    **{
        f"ensemble_{i}": pseudodata_weights[name].to_numpy()
        for i, name in enumerate(n_init_weights)
    },
    **{
        f"bootstrap_data_{i}": pseudodata_weights[name].to_numpy()
        for i, name in enumerate(dbootstrap_weights)
    },
    **{
        f"bootstrap_mc_{i}": pseudodata_weights[name].to_numpy()
        for i, name in enumerate(mcbootstrap_train_weights)
    },
    "trackEffMain": pseudodata_weights["weights_trackEffMain"].to_numpy(),
    "trackEffJet": pseudodata_weights["weights_trackEffJet"].to_numpy(),
    "trackFake": pseudodata_weights["weights_trackFake"].to_numpy(),
    "trackPtScale": pseudodata_weights["weights_trackPtScale"].to_numpy(),
    "muCalID": pseudodata_weights["weights_muCalID"].to_numpy(),
    "muCalMS": pseudodata_weights["weights_muCalMS"].to_numpy(),
    "muCalResBias": pseudodata_weights["weights_muCalResBias"].to_numpy(),
    "muCalScale": pseudodata_weights["weights_muCalScale"].to_numpy(),
    "muEffReco": pseudodata_weights["weights_muEffReco"].to_numpy(),
    "muEffIso": pseudodata_weights["weights_muEffIso"].to_numpy(),
    "muEffTrack": pseudodata_weights["weights_muEffTrack"].to_numpy(),
    "muEffTrig": pseudodata_weights["weights_muEffTrig"].to_numpy(),
    "pileup": pseudodata_weights["weights_pileup"].to_numpy(),
    "lumi": pseudodata_weights["weights_lumi"].to_numpy(),
    "dd": pseudodata_weights["weights_dd"].to_numpy(),
    "target_dd": pseudodata_weights["target_dd"].to_numpy(),
    "hvhad": pseudodata_weights["weights_hvhad"].to_numpy(),
}
truth_pd_weights = {
    "truthpd": truth_pseudodata_weights["weights_nominal"].to_numpy(),
}
pd_hv_weights = {
    "hv": pseudodata_hv_weights["weights_hv"].to_numpy(),
}

In [6]:
bins = np.logspace(-6, -1, 60)
print(bins)

[1.00000000e-06 1.21547425e-06 1.47737765e-06 1.79571449e-06
 2.18264473e-06 2.65294846e-06 3.22459055e-06 3.91940677e-06
 4.76393801e-06 5.79044398e-06 7.03813555e-06 8.55467254e-06
 1.03979842e-05 1.26384820e-05 1.53617495e-05 1.86718109e-05
 2.26951054e-05 2.75853162e-05 3.35292415e-05 4.07539297e-05
 4.95353521e-05 6.02089449e-05 7.31824222e-05 8.89513497e-05
 1.08118075e-04 1.31414736e-04 1.59731228e-04 1.94149195e-04
 2.35983347e-04 2.86831681e-04 3.48636523e-04 4.23758716e-04
 5.15067808e-04 6.26051657e-04 7.60949669e-04 9.24914728e-04
 1.12421004e-03 1.36644835e-03 1.66088278e-03 2.01876025e-03
 2.45375111e-03 2.98247129e-03 3.62511705e-03 4.40623643e-03
 5.35566692e-03 6.50967523e-03 7.91234262e-03 9.61724871e-03
 1.16895182e-02 1.42083083e-02 1.72698329e-02 2.09910372e-02
 2.55140652e-02 3.10116893e-02 3.76939098e-02 4.58159767e-02
 5.56881399e-02 6.76875001e-02 8.22724134e-02 1.00000000e-01]


In [ ]:
# Run computations with dask
cluster = distributed.LocalCluster(
    n_workers=100,
    threads_per_worker=2,
    memory_limit="auto",
)
client = distributed.Client(cluster)

print("Histogramming Madgraph MC sample")
truth_mc_hists = eec.run_eec_workflow_parallel(
    madgraph_path,
    "OmniTree",
    pd_weights,
    bins,
    chunk_size=1000,
)
print("Histogramming HV sample")
truth_hv_hists = eec.run_eec_workflow_parallel(
    hv_path,
    "OmniTree",
    pd_hv_weights,
    bins,
    chunk_size=1000,
)
print("Histogramming Truth Pseudodata sample")
truth_pd_hists = eec.run_eec_workflow_parallel(
    pseudodata_path,
    "OmniTree",
    truth_pd_weights,
    bins,
    chunk_size=1000,
)

client.close()
cluster.close()

In [ ]:
# Pickle the resulting histograms
hist_dir = "./hist_storage"
if not os.path.exists(hist_dir):
    os.makedirs(hist_dir)

with open(os.path.join(hist_dir, "pd_eec_hists.pkl"), "wb") as f:
    pickle.dump(
        {
            "truth_mc": truth_mc_hists,
            "truth_hv": truth_hv_hists,
            "truth_pd": truth_pd_hists,
        },
        f,
    )

### Plotting

In [ ]:
# Load the histograms
hist_dir = "./hist_storage"
with open(os.path.join(hist_dir, "pd_eec_hists.pkl"), "rb") as f:
    all_hists = pickle.load(f)

measurement_hists = {**all_hists["truth_mc"], **all_hists["truth_hv"]}
truth_pd_hists = all_hists["truth_pd"]

In [ ]:
# Plot comparison to truth pseudodata (no uncertainties)
import visualize as vis
comp = vis.compare_to_target(
    measurement_hists,
    truth_pd_hists,
    log_xscale=True,
    linear_yscale=False,
    ylabel="EEC",
    xlabel="z",
    normalize=True,
    rlab="All tracks, $p_T > 500$ MeV",
)

In [ ]:
# Plot pseudo-measurement and uncertainty budget
import visualize as vis

xsec, budget, cov = vis.plot_measurement_with_uncertainties(
    measurement_hists,
    truth_pd_hists,
    figsize=(6.4, 4.8),
    color="blue",
    linear_yscale=False,
    ylabel="EEC",
    xlabel="z",
    normalize=True,
    llab="Simulation Preliminary",
    rlab="All tracks, $p_T > 500$ MeV",
    simple_corr_labels=True,
    do_chi2_test=True,
    ylim=(1e-4, 1e0),
)
xsec.show()
budget.show()
cov.show()

## Measurement with data

In [ ]:
# Load all weights from the Omnifold data measurement
of_data = pd.read_hdf(
    "../weight_storage/zjets-v4/data-weights.h5",
    key="weights",
    mode="r",
)
of_data_hv = pd.read_hdf(
    "../weight_storage/zjets-v4/data-weights.h5",
    key="hv_weights",
    mode="r",
)

In [ ]:
# Create dictionary describing all of the histograms needed for the data measurement
n_init_weights = [name for name in of_data.keys() if "weights_ensemble_" in name]
dbootstrap_weights = [
    name for name in of_data.keys() if "weights_bootstrap_data_" in name
]
mcbootstrap_weights = [
    name for name in of_data.keys() if "weights_bootstrap_mc_" in name
]
truth_mc_weights = {
    "nominal": of_data["weights_nominal"].to_numpy()[truth_pass200 == 1],
    **{
        f"ensemble_{i}": of_data[name].to_numpy()[truth_pass200 == 1]
        for i, name in enumerate(n_init_weights)
    },
    **{
        f"bootstrap_data_{i}": of_data[name].to_numpy()[truth_pass200 == 1]
        for i, name in enumerate(dbootstrap_weights)
    },
    **{
        f"bootstrap_mc_{i}": of_data[name].to_numpy()[truth_pass200 == 1]
        for i, name in enumerate(mcbootstrap_weights)
    },
    "trackEffMain": of_data["weights_trackEffMain"].to_numpy()[truth_pass200 == 1],
    "trackEffJet": of_data["weights_trackEffJet"].to_numpy()[truth_pass200 == 1],
    "trackFake": of_data["weights_trackFake"].to_numpy()[truth_pass200 == 1],
    "trackPtScale": of_data["weights_trackPtScale"].to_numpy()[truth_pass200 == 1],
    "muCalID": of_data["weights_muCalID"].to_numpy()[truth_pass200 == 1],
    "muCalMS": of_data["weights_muCalMS"].to_numpy()[truth_pass200 == 1],
    "muCalResBias": of_data["weights_muCalResBias"].to_numpy()[truth_pass200 == 1],
    "muCalScale": of_data["weights_muCalScale"].to_numpy()[truth_pass200 == 1],
    "muEffReco": of_data["weights_muEffReco"].to_numpy()[truth_pass200 == 1],
    "muEffIso": of_data["weights_muEffIso"].to_numpy()[truth_pass200 == 1],
    "muEffTrack": of_data["weights_muEffTrack"].to_numpy()[truth_pass200 == 1],
    "muEffTrig": of_data["weights_muEffTrig"].to_numpy()[truth_pass200 == 1],
    # "pileup": of_data["weights_pileup"].to_numpy()[truth_pass200 == 1],
    "lumi": of_data["weights_lumi"].to_numpy()[truth_pass200 == 1],
    "dd": of_data["weights_dd"].to_numpy()[truth_pass200 == 1],
    "target_dd": of_data["target_dd"].to_numpy()[truth_pass200 == 1],
}
hv_weights = {
    "hv": of_data_hv["weights_hv"].to_numpy()[truth_pass200_hv == 1],
}
truth_madgraph_weights = {
    "madgraph": mg_weights["nominal"][truth_pass200 == 1],
    "weights_theoryQCD": mg_weights["weights_theoryQCD"][truth_pass200 == 1],
    "weights_theoryPDF": mg_weights["weights_theoryPDF"][truth_pass200 == 1],
    "weights_theoryAlphaS": mg_weights["weights_theoryAlphaS"][truth_pass200 == 1],
    "weights_theoryPSsoft": mg_weights["weights_theoryPSsoft"][truth_pass200 == 1],
    "weights_theoryPSjet": mg_weights["weights_theoryPSjet"][truth_pass200 == 1],
    "weights_theoryPSscale": mg_weights["weights_theoryPSscale"][truth_pass200 == 1],
    "weights_theoryMPI": mg_weights["weights_theoryMPI"][truth_pass200 == 1],
}
truth_sherpa_weights = {
    "sherpa": sherpa_weights["nominal"][truth_pass200_sherpa == 1],
    "weights_theoryQCD": sherpa_weights["weights_theoryQCD"][truth_pass200_sherpa == 1],
    "weights_theoryPDF": sherpa_weights["weights_theoryPDF"][truth_pass200_sherpa == 1],
    "weights_theoryAlphaS": sherpa_weights["weights_theoryAlphaS"][
        truth_pass200_sherpa == 1
    ],
}

In [ ]:
bins = np.logspace(-6, -1, 60)
print(bins)

In [ ]:
# Run computations with dask
n_cores = os.cpu_count()
cluster = distributed.LocalCluster(
    n_workers=100,
    threads_per_worker=2,
    memory_limit="auto",
)
client = distributed.Client(cluster)

# Histogram the Madgraph MC sample
print("Histogramming Madgraph MC sample")
truth_mc_hists = eec.run_eec_workflow_parallel(
    mg_mc_path,
    "OmniTree",
    truth_pass200,
    truth_mc_weights,
    bins,
    get_truth=True,
    chunk_size=1000,
)
print("Histogramming HV sample")
truth_hv_hists = eec.run_eec_workflow_parallel(
    hv_mc_path,
    "OmniTree",
    truth_pass200_hv,
    hv_weights,
    bins,
    get_truth=True,
    chunk_size=1000,
)
print("Histogramming Madgraph prediction")
truth_madgraph_hists = eec.run_eec_workflow_parallel(
    mg_mc_path,
    "OmniTree",
    truth_pass200,
    truth_madgraph_weights,
    bins,
    get_truth=True,
    chunk_size=1000,
)
print("Histogramming Sherpa prediction")
truth_sherpa_hists = eec.run_eec_workflow_parallel(
    sherpa_mc_path,
    "OmniTree",
    truth_pass200_sherpa,
    truth_sherpa_weights,
    bins,
    get_truth=True,
    chunk_size=1000,
    max_events=5000000,
)

client.close()
cluster.close()

In [ ]:
# Pickle the resulting histograms
hist_dir = "./hist_storage"
with open(os.path.join(hist_dir, "data_eec_hists.pkl"), "wb") as f:
    pickle.dump(
        {
            "truth_mc": truth_mc_hists,
            "truth_hv": truth_hv_hists,
            "truth_madgraph": truth_madgraph_hists,
            "truth_sherpa": truth_sherpa_hists,
        },
        f,
    )

## Plotting

In [ ]:
# Load the histograms
hist_dir = "./hist_storage"
with open(os.path.join(hist_dir, "data_eec_hists.pkl"), "rb") as f:
    all_hists = pickle.load(f)

measurement_hists = {**all_hists["truth_mc"], **all_hists["truth_hv"]}
truth_madgraph_hists = all_hists["truth_madgraph"]
truth_sherpa_hists = all_hists["truth_sherpa"]

In [ ]:
# Plot data measurement and uncertainty budget
xsec, budget, cov = vis.plot_measurement_with_uncertainties(
    measurement_hists,
    truth_madgraph_hists,
    truth_sherpa_hists,
    measured_label="Data",
    target_key="madgraph",
    target_label="MadGraph",
    target2_key="sherpa",
    target2_label="Sherpa",
    data_measurement_mode=True,
    color="black",
    linear_yscale=False,
    ylabel="EEC",
    xlabel="z",
    normalize=True,
    rlab="All tracks, $p_T > 500$ MeV",
    ylim=(1e-3, 1e0),
    rlim=(0.7, 1.3),
    simple_corr_labels=True,
)
xsec.savefig("plot_storage/data_eec_xsec.pdf", dpi=300)
xsec.show()
budget.savefig("plot_storage/data_eec_budget.pdf", dpi=300)
budget.show()
cov.savefig("plot_storage/data_eec_cov.pdf", dpi=300)
cov.show()